In [15]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
# import geopandas as gpd
# from shapely.geometry import Point
import glob

from tqdm import tqdm
import os
import shutil
import tempfile
import logging
import geopandas as gpd
import rioxarray as rxr

In [16]:
# LDD direction convention (PCRaster):
# 7 8 9
# 4 5 6
# 1 2 3
# For each direction, what is the (row_offset, col_offset) of the downstream neighbour?
# e.g. direction 1 means flow goes to bottom-left → downstream is at (+1, -1)

# Inverse: for each neighbour position, which LDD value points TO the current cell?
# neighbour at (-1, -1) has direction 3 if it flows to us
# neighbour at (-1,  0) has direction 2
# neighbour at (-1, +1) has direction 1
# neighbour at ( 0, -1) has direction 6
# neighbour at ( 0, +1) has direction 4
# neighbour at (+1, -1) has direction 9
# neighbour at (+1,  0) has direction 8
# neighbour at (+1, +1) has direction 7

# (row_offset, col_offset) -> LDD value that points toward current cell
UPSTREAM_DIRS = {
    (-1, -1): 3,
    (-1,  0): 2,
    (-1, +1): 1,
    ( 0, -1): 6,
    ( 0, +1): 4,
    ( 1, -1): 9,
    ( 1,  0): 8,
    ( 1, +1): 7,
}

def cutmaps_own(ldd_array, lon_array, lat_array, station_lon, station_lat):
    """
    Trace upstream from station pixel through LDD to get catchment mask.
    Returns a 2D boolean array (same shape as ldd_array).
    """
    # find nearest pixel to station coordinates
    col = np.argmin(np.abs(lon_array - station_lon))
    row = np.argmin(np.abs(lat_array - station_lat))
    # print(f"  Station pixel: row={row}, col={col}, lon={lon_array[col]:.3f}, lat={lat_array[row]:.3f}")

    nrows, ncols = ldd_array.shape
    mask = np.zeros((nrows, ncols), dtype=bool)

    # BFS upstream flood-fill
    queue = [(row, col)]
    mask[row, col] = True

    while queue:
        r, c = queue.pop()
        for (dr, dc), upstream_val in UPSTREAM_DIRS.items():
            nr, nc = r + dr, c + dc
            if 0 <= nr < nrows and 0 <= nc < ncols:
                if not mask[nr, nc] and ldd_array[nr, nc] == upstream_val:
                    mask[nr, nc] = True
                    queue.append((nr, nc))

    return mask

In [17]:
BASE_FILE = Path.cwd () / "GloFASv5_stations_metadata_calfunction_KGE_JSD_20March2026_final.csv"
# Folder with static attributes
DIR_STATIC = Path("/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/static_maps/GloFASv5_staticmaps_consolidated_March2026/GloFASv5_static_maps_reanalysis/")
# Folder with parameter attributes
DIR_PARS = Path("/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/static_maps/GloFASv5_parametermaps/")

# Base File
glofas5_base_info = pd.read_csv(BASE_FILE)

In [18]:
# station list
cal_stations = glofas5_base_info[["LISFLOOD_X","LISFLOOD_Y", "ID"]].values
if not (DIR_STATIC / "cal_stations.txt").exists():
    with open(DIR_STATIC / "cal_stations.txt", "w") as f:
        for x, y, id_ in cal_stations:
            f.write(f"{x:.3f}  {y:.3f}\t{int(id_)}\n")

In [19]:
# load GloFASv5 Parameters
glofas_pars = xr.open_mfdataset(sorted(glob.glob(str(DIR_PARS / "*.nc"))), combine="by_coords")
glofas_pars = glofas_pars.load()

### Load ldd & uparea

In [20]:
# Load LDD file
ldd_nc = DIR_STATIC / "ldd_repaired.nc"
ds_ldd = xr.open_dataset(ldd_nc)
print("\n=== ldd_repaired.nc ===")
print(ds_ldd)
print("\n=== Variables & dtypes ===")
for var in ds_ldd.data_vars:
    print(f"{var}: {ds_ldd[var].dtype}")
ds_ldd.close()

# Load upArea file
uparea_nc = DIR_STATIC / "upArea_repaired_correctedmetadata_3000.nc"
ds_uparea = xr.open_dataset(uparea_nc)
print("\n=== upArea_repaired_correctedmetadata_3000.nc ===")
print(ds_uparea)
print("\n=== Variables & dtypes ===")
for var in ds_uparea.data_vars:
    print(f"{var}: {ds_uparea[var].dtype}")
ds_uparea.close()


=== ldd_repaired.nc ===
<xarray.Dataset> Size: 86MB
Dimensions:  (lat: 3000, lon: 7200)
Coordinates:
  * lat      (lat) float64 24kB 89.97 89.92 89.88 89.82 ... -59.88 -59.92 -59.97
  * lon      (lon) float64 58kB -180.0 -179.9 -179.9 ... 179.9 179.9 180.0
Data variables:
    crs      |S1 1B ...
    Band1    (lat, lon) float32 86MB ...
Attributes:
    CDI:                        Climate Data Interface version 1.9.10 (https:...
    Conventions:                CF-1.5
    GDAL_AREA_OR_POINT:         Area
    GDAL_PCRASTER_VALUESCALE:   VS_LDD
    GDAL:                       GDAL 3.2.1, released 2020/12/29
    CDO:                        Climate Data Operators version 1.9.10 (https:...
    history:                    Fri Dec 10 11:49:28 2021: ncks -A -v lat,lon ...
    history_of_appended_files:  Fri Dec 10 11:49:28 2021: Appended file ldd_O...
    NCO:                        netCDF Operators version 4.9.7 (Homepage = ht...

=== Variables & dtypes ===
crs: |S1
Band1: float32

=== upArea_r

### loop over all catchments & assign glacier coverage

In [21]:
"""
catchment_parameter_attributes.py
-----------------------------------
Extracts GloFASv5 calibration parameter values per catchment
using BFS upstream tracer on LDD.
"""

import time
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

# =============================================================================
# STEP 0 — load LDD + parameter maps
# =============================================================================

print("Loading LDD...")
ldd_array = np.squeeze(ds_ldd['Band1'].values)
lon_array = ds_ldd.lon.values
lat_array = ds_ldd.lat.values
print(f"LDD shape: {ldd_array.shape}")

print("Loading parameter maps...")
par_arrays = {}
for par in glofas_pars.data_vars:
    arr = np.squeeze(glofas_pars[par].values)
    if arr.shape == ldd_array.shape:
        par_arrays[par] = arr.astype(float)
        print(f"  Loaded {par}  shape: {arr.shape}")
    else:
        print(f"  WARNING: {par} shape {arr.shape} does not match LDD, skipping")
print(f"\nLoaded {len(par_arrays)} parameter maps.")

# =============================================================================
# STEP 1+2 — loop over stations: BFS mask → parameter value
# =============================================================================

cal_stations = glofas5_base_info[["LISFLOOD_X", "LISFLOOD_Y", "ID"]].values
results = []
t_total = time.time()

for x, y, station_id in tqdm(cal_stations, desc="Processing stations"):
    station_id = int(station_id)
    row = {"ID": station_id}

    # BFS catchment mask
    mask = cutmaps_own(ldd_array, lon_array, lat_array, x, y)
    rows_idx, cols_idx = np.where(mask)

    if len(rows_idx) == 0:
        for par in par_arrays:
            row[par] = np.nan
        results.append(row)
        continue

    r_min, r_max = rows_idx.min(), rows_idx.max()
    c_min, c_max = cols_idx.min(), cols_idx.max()
    mask_cut = mask[r_min:r_max+1, c_min:c_max+1]

    # loop over parameters — direct numpy slice, same grid as LDD
    for par, arr in par_arrays.items():
        par_cut = arr[r_min:r_max+1, c_min:c_max+1]

        vals = par_cut[mask_cut].astype(float)
        vals = vals[np.isfinite(vals)]

        if len(vals) == 0:
            row[par] = np.nan
            continue

        row[par] = float(vals[0])  # all pixels same value, take first valid one

    results.append(row)

print(f"\nDone! {len(results)} stations in {time.time()-t_total:.1f}s")

# =============================================================================
# STEP 3 — merge and save
# =============================================================================

df_pars = pd.DataFrame(results)
df_pars["ID"] = df_pars["ID"].astype(int)
glofas5_base_info["ID"] = glofas5_base_info["ID"].astype(int)

# deduplicate base info before merging to avoid fan-out
glofas5_base_info_dedup = glofas5_base_info.drop_duplicates(subset="ID")
glofas5_enriched = glofas5_base_info_dedup.merge(df_pars, on="ID", how="left")

print(f"Final shape: {glofas5_enriched.shape}")
print(glofas5_enriched.head())

Loading LDD...
LDD shape: (3000, 7200)
Loading parameter maps...
  Loaded CalChanMan1  shape: (3000, 7200)
  Loaded CalChanMan3  shape: (3000, 7200)
  Loaded GwLoss  shape: (3000, 7200)
  Loaded GwPercValue  shape: (3000, 7200)
  Loaded LZThreshold  shape: (3000, 7200)
  Loaded LakeMultiplier  shape: (3000, 7200)
  Loaded LowerZoneTimeConstant  shape: (3000, 7200)
  Loaded PowerPrefFlow  shape: (3000, 7200)
  Loaded SnowMeltCoef  shape: (3000, 7200)
  Loaded TransSub  shape: (3000, 7200)
  Loaded UpperZoneTimeConstant  shape: (3000, 7200)
  Loaded b_Xinanjiang  shape: (3000, 7200)

Loaded 12 parameter maps.


Processing stations: 100%|██████████| 5379/5379 [05:56<00:00, 15.10it/s]


Done! 5379 stations in 356.2s
Final shape: (5379, 34)
   ID                      name basin        river  provider iso  \
0   4                     Barby  Elbe         Elbe      1004  DE   
1   5  Wittenberg / Lutherstadt  Elbe         Elbe      1004  DE   
2   6                Malliss OP  Elbe  Muritz-Elde      1004  DE   
3   9               Rathenow UP  Elbe  Lower Havel      1004  DE   
4  11            Calbe Grizehne  Elbe        Saale      1004  DE   

   DrainageArea_prov  DrainageArea_LDD        lat       long  ...    GwLoss  \
0           94060.00          93699.47  51.984836  11.882244  ...  0.000000   
1           61879.00          60938.53  51.856531  12.646309  ...  0.198080   
2            2879.00           3161.95  53.190627  11.345254  ...  0.255436   
3           19288.46          18271.30  52.607446  12.321014  ...  0.056843   
4           23719.00          23560.46  51.916414  11.812211  ...  0.000000   

   GwPercValue LZThreshold LakeMultiplier LowerZoneTimeConsta

In [22]:
# save the results
glofas5_enriched.to_csv("GloFASv5_Stations_Overview_260223/GloFASv5_catchments_parameters_comprehensive.csv", index=False)